## Read and load the csv file 

In [168]:
import pandas as pd 
import numpy as np 

df=pd.read_csv("data/test_weather_data.csv")

df.head(10)

,date,avg_temp,min_temp,max_temp,rainfall_mm,humidity,weather_notes
0,01-01-2016,3.0,-5.0,10.4,0.15,57,sunny
1,2016-01-02,5.9,2.1,13.8,0.09,95,clear sky
2,01-03-2016,7.8,NaN,8.9,3.47,58,humid
3,2016-01-04,11.0,1.4,13.3,3.03,65,sudden shower
4,2016-01-05,NaN,-0.5,8.1,1.42,86,sunny
5,2016-01-06,17.2,12.8,17.4,23.78,56,humid
6,2016-01-07,22.5,19.4,26.8,1.34,90%,storm
7,2016-01-08,11.6,8.2,15.3,0.43,71,DRY HEAT
8,2016-01-09,21.6,19.8,31.0,0.79,56,windy
9,2016-01-10,30.7C,23.6,38.8,0.84mm,52,blizzard


## Early report of dataset

In [169]:
#dataset report before cleaning 
early_report = {
    "rows": len(df),
    "missing_values": df.isna().sum(),
    "duplicates": df.duplicated().sum()
}

print(early_report)

{'rows': 3813, 'missing_values': date               0
avg_temp         308
min_temp         185
max_temp         158
rainfall_mm        0
humidity           0
weather_notes     95
dtype: int64, 'duplicates': np.int64(100)}


## Parse mixed date formats

In [170]:
df["date"] = pd.to_datetime(df["date"], errors="coerce", format="mixed",  yearfirst=True) #handles date formats(with mixed pandas interprets each row   ), rolls that fails parsing becomes NaT


print(df)
df[df["date"].isna()] #checks missing dates 

           date avg_temp  min_temp  max_temp rainfall_mm humidity  \
0    2016-01-01      3.0      -5.0      10.4        0.15       57   
1    2016-01-02      5.9       2.1      13.8        0.09       95   
2    2016-01-03      7.8       NaN       8.9        3.47       58   
3    2016-01-04     11.0       1.4      13.3        3.03       65   
4    2016-01-05      NaN      -0.5       8.1        1.42       86   
...         ...      ...       ...       ...         ...      ...   
3808 2018-11-05      9.0       8.2      17.1        1.55       39   
3809 2024-06-18     13.2       8.9      17.9        3.51       99   
3810 2021-11-12     31.6      30.9      39.9      5.02mm      38%   
3811 2022-10-27     16.1      10.2      26.2        0.35       96   
3812 2026-01-09      5.6      -1.0      18.1       trace       88   

      weather_notes  
0             sunny  
1         clear sky  
2             humid  
3     sudden shower  
4             sunny  
...             ...  
3808       dry he

,date,avg_temp,min_temp,max_temp,rainfall_mm,humidity,weather_notes


## Strip units from columns

In [171]:
#Example,Remove C, mm, %. (Note: We use astype(str) to force convert column values to strings to be able to strip units attached  )

df["avg_temp"] = df["avg_temp"].astype(str).str.replace("C", "", regex=False) #regex =false treats it as a text and does not remove all characters 
df["rainfall_mm"] = df["rainfall_mm"].astype(str).str.replace("mm", "", regex=False)
df["humidity"] = df["humidity"].astype(str).str.replace("%", "", regex=False)


#Remove white spaces 

df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col) #this is specifically for text columns 

print(df)


           date avg_temp  min_temp  max_temp rainfall_mm humidity  \
0    2016-01-01      3.0      -5.0      10.4        0.15       57   
1    2016-01-02      5.9       2.1      13.8        0.09       95   
2    2016-01-03      7.8       NaN       8.9        3.47       58   
3    2016-01-04     11.0       1.4      13.3        3.03       65   
4    2016-01-05      NaN      -0.5       8.1        1.42       86   
...         ...      ...       ...       ...         ...      ...   
3808 2018-11-05      9.0       8.2      17.1        1.55       39   
3809 2024-06-18     13.2       8.9      17.9        3.51       99   
3810 2021-11-12     31.6      30.9      39.9        5.02       38   
3811 2022-10-27     16.1      10.2      26.2        0.35       96   
3812 2026-01-09      5.6      -1.0      18.1       trace       88   

      weather_notes  
0             sunny  
1         clear sky  
2             humid  
3     sudden shower  
4             sunny  
...             ...  
3808       dry he

## Convert columns to numeric

In [172]:
#Invalid values becomes NaN

numeric_cols = ["avg_temp", "min_temp", "max_temp", "rainfall_mm", "humidity"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

    print(df)

           date  avg_temp  min_temp  max_temp rainfall_mm humidity  \
0    2016-01-01       3.0      -5.0      10.4        0.15       57   
1    2016-01-02       5.9       2.1      13.8        0.09       95   
2    2016-01-03       7.8       NaN       8.9        3.47       58   
3    2016-01-04      11.0       1.4      13.3        3.03       65   
4    2016-01-05       NaN      -0.5       8.1        1.42       86   
...         ...       ...       ...       ...         ...      ...   
3808 2018-11-05       9.0       8.2      17.1        1.55       39   
3809 2024-06-18      13.2       8.9      17.9        3.51       99   
3810 2021-11-12      31.6      30.9      39.9        5.02       38   
3811 2022-10-27      16.1      10.2      26.2        0.35       96   
3812 2026-01-09       5.6      -1.0      18.1       trace       88   

      weather_notes  
0             sunny  
1         clear sky  
2             humid  
3     sudden shower  
4             sunny  
...             ...  
3808 

## Flag Suspicious Data 

In [173]:
#flags suspicious data but not delete 
df["temp_flag"] = (
    (df["avg_temp"] < -50) |  #temperature outliers 
    (df["avg_temp"] > 60)
)
df["flag_min_max_swapped"] = df["min_temp"] > df["max_temp"]
df["rain_flag"] = df["rainfall_mm"] > 300  #rain outliers 
df["humidity_flag"] = (df["humidity"] < 0) | (df["humidity"] > 100)

print(df)


           date  avg_temp  min_temp  max_temp  rainfall_mm  humidity  \
0    2016-01-01       3.0      -5.0      10.4         0.15        57   
1    2016-01-02       5.9       2.1      13.8         0.09        95   
2    2016-01-03       7.8       NaN       8.9         3.47        58   
3    2016-01-04      11.0       1.4      13.3         3.03        65   
4    2016-01-05       NaN      -0.5       8.1         1.42        86   
...         ...       ...       ...       ...          ...       ...   
3808 2018-11-05       9.0       8.2      17.1         1.55        39   
3809 2024-06-18      13.2       8.9      17.9         3.51        99   
3810 2021-11-12      31.6      30.9      39.9         5.02        38   
3811 2022-10-27      16.1      10.2      26.2         0.35        96   
3812 2026-01-09       5.6      -1.0      18.1          NaN        88   

      weather_notes  temp_flag  flag_min_max_swapped  rain_flag  humidity_flag  
0             sunny      False                 False  

## Fix swapped min/max temperatures with mask

In [174]:
# creates a mask (boolean series)
mask = df["min_temp"] > df["max_temp"] #returns True or False if value swapped 


df.loc[mask, ["min_temp", "max_temp"]] = (   #iteerates through the rows and points out where min is > max and then reverses the column order ...
    df.loc[mask, ["max_temp", "min_temp"]].values  #(.values means converting the selection to a numpy array so pandas assigns values correctly and not by column name . )
)

print(mask.sum()) #check total fixed

99


## Replace impossible values with NaN

In [175]:
# Instead of deleting rows we mark them as missing .. Note:impossible values can be deleted for very strict cleaning 

df.loc[(df["avg_temp"] < -50) | (df["avg_temp"] > 60), "avg_temp"] = np.nan  #temperature limit 
df.loc[df["rainfall_mm"] > 300, "rainfall_mm"] = np.nan #rainfall limits 
df.loc[(df["humidity"] < 0) | (df["humidity"] > 100), "humidity"] = np.nan #humidity limits 

print(df)

           date  avg_temp  min_temp  max_temp  rainfall_mm  humidity  \
0    2016-01-01       3.0      -5.0      10.4         0.15      57.0   
1    2016-01-02       5.9       2.1      13.8         0.09      95.0   
2    2016-01-03       7.8       NaN       8.9         3.47      58.0   
3    2016-01-04      11.0       1.4      13.3         3.03      65.0   
4    2016-01-05       NaN      -0.5       8.1         1.42      86.0   
...         ...       ...       ...       ...          ...       ...   
3808 2018-11-05       9.0       8.2      17.1         1.55      39.0   
3809 2024-06-18      13.2       8.9      17.9         3.51      99.0   
3810 2021-11-12      31.6      30.9      39.9         5.02      38.0   
3811 2022-10-27      16.1      10.2      26.2         0.35      96.0   
3812 2026-01-09       5.6      -1.0      18.1          NaN      88.0   

      weather_notes  temp_flag  flag_min_max_swapped  rain_flag  humidity_flag  
0             sunny      False                 False  

## Remove duplicate rows

In [176]:
df = df.drop_duplicates()

#drops duplicates by date 
df = df.drop_duplicates(subset=["date"])

df.duplicated().sum()

np.int64(0)

## Standardize weather notes 

In [ ]:
#Clean capitalization and punctuation.

df["weather_notes"] = (
    df["weather_notes"]
    .astype(str)
    .str.lower()
    .str.replace("!", "", regex=False)
    .str.strip()
)

#Replace empty notes in column "weather notes" to display unknown 
df["weather_notes"] = df["weather_notes"].replace("", "unknown")

print(df)

           date  avg_temp  min_temp  max_temp  rainfall_mm  humidity  \
0    2016-01-01       3.0      -5.0      10.4         0.15      57.0   
1    2016-01-02       5.9       2.1      13.8         0.09      95.0   
2    2016-01-03       7.8       NaN       8.9         3.47      58.0   
3    2016-01-04      11.0       1.4      13.3         3.03      65.0   
4    2016-01-05       NaN      -0.5       8.1         1.42      86.0   
...         ...       ...       ...       ...          ...       ...   
3708 2026-02-25       NaN      -3.8       8.3         3.99      44.0   
3709 2026-02-26      11.7       4.3      20.6         3.49      35.0   
3710 2026-02-27      21.0      16.9      21.2          NaN      76.0   
3711 2026-02-28      17.9      16.2      18.9         4.88      55.0   
3712 2026-03-01      18.7      15.3      19.4        11.39      49.0   

      weather_notes  temp_flag  flag_min_max_swapped  rain_flag  humidity_flag  
0             sunny      False                 False  

## Final validation checks

In [178]:
print(df.describe())

#print(df.isna().sum())

#print(df.head())

                             date     avg_temp     min_temp     max_temp  \
count                        3645  3352.000000  3467.000000  3496.000000   
mean   2021-02-01 18:49:52.592592    12.067601     6.888261    17.151916   
min           2016-01-01 00:00:00   -30.200000   -39.600000   -23.700000   
25%           2018-07-19 00:00:00     5.500000    -0.200000    10.200000   
50%           2021-02-05 00:00:00    12.100000     6.900000    17.200000   
75%           2023-08-19 00:00:00    18.800000    14.000000    24.125000   
max           2026-03-01 00:00:00    45.900000    41.200000    53.800000   
std                           NaN     9.977889    10.368307    10.386590   

       rainfall_mm     humidity  
count  3472.000000  3542.000000  
mean      3.032964    64.382270  
min       0.000000    30.000000  
25%       0.940000    47.000000  
50%       2.100000    64.000000  
75%       4.172500    82.000000  
max      23.780000    99.000000  
std       2.952167    20.297981  


In [179]:
print(df.head())

        date  avg_temp  min_temp  max_temp  rainfall_mm  humidity  \
0 2016-01-01       3.0      -5.0      10.4         0.15      57.0   
1 2016-01-02       5.9       2.1      13.8         0.09      95.0   
2 2016-01-03       7.8       NaN       8.9         3.47      58.0   
3 2016-01-04      11.0       1.4      13.3         3.03      65.0   
4 2016-01-05       NaN      -0.5       8.1         1.42      86.0   

   weather_notes  temp_flag  flag_min_max_swapped  rain_flag  humidity_flag  
0          sunny      False                 False      False          False  
1      clear sky      False                 False      False          False  
2          humid      False                 False      False          False  
3  sudden shower      False                 False      False          False  
4          sunny      False                 False      False          False  


## Data quality report 

In [180]:
report = {
    "rows": len(df),
    "missing_values": df.isna().sum(),
    "duplicates": df.duplicated().sum()
}

print(report)

{'rows': 3645, 'missing_values': date                      0
avg_temp                293
min_temp                178
max_temp                149
rainfall_mm             173
humidity                103
weather_notes            91
temp_flag                 0
flag_min_max_swapped      0
rain_flag                 0
humidity_flag             0
dtype: int64, 'duplicates': np.int64(0)}
